# 00 · Setup and free analyses (no OpenAI cost)

Run this first. Nothing here calls the paid API.

| Notebook | What | Cost |
|---|---|---|
| **00** (this) | setup, data preparation, re-analysis of the reported statistics, lexicon/VADER baselines, annotation sheets, cost plan | free |
| **01** | **Core reviewer responses**: ablation C0–C3 + RDS dimensions | ≈ $1.5 left (C0–C3 already done) |
| **02** | Speculative RAG (C4) | ≈ $5.5 left (~40% done) |
| **03** | Robustness extras: original replication, stability, counterfactual focus, external dataset | ≈ $9 |
| **04** | Human annotation agreement, supervised/transformer baselines, final analysis and figures | free |

Every paid notebook has its own `BUDGET_USD` cap and stops cleanly when the cap or your OpenAI credit is reached.

## Setup

In [ ]:
import os, sys
from pathlib import Path

def find_revision_dir() -> Path:
    here = Path.cwd().resolve()
    for cand in [here, here.parent, here / "Revision", here.parent / "Revision"]:
        if (cand / "code" / "config.py").exists():
            return cand
    raise FileNotFoundError("Open this notebook from Behavioral_Economics/Revision/notebooks/")

REVISION_DIR = find_revision_dir()
PROJECT_DIR = REVISION_DIR.parent
os.environ["REVISION_DIR"] = str(REVISION_DIR)
os.environ.setdefault("DATASET_DIR", str(PROJECT_DIR / "Dataset"))
os.environ.setdefault("MAX_WORKERS", "8")   # parallel API calls; lower to 2-4 if you hit rate limits
if str(REVISION_DIR / "code") not in sys.path:
    sys.path.insert(0, str(REVISION_DIR / "code"))
print("Revision folder:", REVISION_DIR, "| Python", sys.version.split()[0])

In [ ]:
# Installs any missing packages into THIS kernel (uv, venv or conda). Does nothing if all are present.
import importlib.util, shutil, subprocess
CORE = {"openai": "openai", "pandas": "pandas", "numpy": "numpy", "scipy": "scipy", "sklearn": "scikit-learn",
        "statsmodels": "statsmodels", "faiss": "faiss-cpu", "pypdf": "pypdf",
        "langchain_text_splitters": "langchain-text-splitters", "openpyxl": "openpyxl",
        "vaderSentiment": "vaderSentiment", "krippendorff": "krippendorff", "lingua": "lingua-language-detector",
        "matplotlib": "matplotlib", "tqdm": "tqdm", "ipywidgets": "ipywidgets"}

def _install(req_file):
    py = sys.executable
    uv = shutil.which("uv") or shutil.which("uv.exe")
    if uv:
        cmd = [uv, "pip", "install", "--python", py, "-r", str(req_file)]
    else:
        if importlib.util.find_spec("pip") is None:
            subprocess.run([py, "-m", "ensurepip", "--upgrade"], check=True)
        cmd = [py, "-m", "pip", "install", "-r", str(req_file)]
    print("Running:", " ".join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    print((r.stdout or "")[-1500:], (r.stderr or "")[-1500:]); r.check_returncode()

missing = [pkg for mod, pkg in CORE.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Installing:", missing); _install(REVISION_DIR / "requirements.txt"); importlib.invalidate_caches()
print("Missing packages:", [p for m, p in CORE.items() if importlib.util.find_spec(m) is None] or "none")

In [ ]:
import importlib, json
import pandas as pd
import config as C                     # also loads ../.env
import pipeline as PL
C.ensure_dirs()
pd.set_option("display.max_colwidth", 110); pd.set_option("display.width", 200)
MOCK = os.environ.get("MOCK_LLM") == "1"   # code testing only - never for results
print("OpenAI key in use :", C.key_fingerprint(), "| loaded from:", C.ENV_LOADED or "environment (no .env found)")
print("If this is not the key you expect: fix Behavioral_Economics/.env, then Restart the kernel and re-run.")

## 1 · Data preparation (R1.1, R4.4)

In [ ]:
import data_prep
data_prep.run()
display(pd.read_csv(C.RESULTS_DIR / "01_data" / "data_flow.csv"))
display(pd.read_csv(C.RESULTS_DIR / "01_data" / "category_pool_vs_sample.csv"))

## 2 · Re-analysis from the paper's reported statistics (R2.5, R3.4, R2.4, R3.3)

In [ ]:
import reported_stats
rs = reported_stats.run()
display(pd.read_csv(C.RESULTS_DIR / "00_reported_stats" / "reanalysis_from_reported_stats.csv"))
print(json.dumps(rs["reanalysis"]["internal_consistency"], indent=2))
print(json.dumps(rs["algebra"], indent=2))

## 3 · Lexicon and VADER baselines (R2.1, R4.6)

In [ ]:
import baselines_classic as BC
for ds in ["pool", "external"]:
    BC.run_pool(ds)
display(pd.read_csv(C.RESULTS_DIR / "05_baselines" / "pool_lexicon_vader_summary.csv"))

## 4 · Human-annotation workbooks (never overwritten once created)

In [ ]:
import annotation
annotation.build_sheets()
print("Coder sheets:", sorted(p.name for p in C.ANNOT_DIR.glob("annotation_coder_*.xlsx")))

## 5 · Spending plan: what is already paid for, and what each part still costs

In [ ]:
display(PL.remaining_summary())
display(PL.remaining_estimate()[["part", "stage", "namespace", "calls_cached", "calls_left", "pct_done", "usd_left_indicative"]])